## LG Aimers 9기 — 시간 검증 모델링 (Colab)

이 노트북은 현재 기준선인 LightGBM을 **과거 시즌 학습 → 다음 시즌 검증** 방식으로 실행하고 결과를 모은다. 기본 설정은 2019~2023으로 학습해 2024를 검증한다.

중요: 현재 LightGBM 기준선은 CPU 모델이다. Colab에서 GPU 런타임을 선택해도 자동으로 GPU를 쓰지 않는다. 대신 데이터/코드는 그대로 두고, 다음 실험에서 GPU를 직접 지원하는 CatBoost/XGBoost를 추가해 앙상블 후보로 비교한다. LightGBM은 1개 폴드가 비교적 짧아 CPU 기준선으로 유지한다.

### 0. 프로젝트 위치 확인

Colab에서는 저장소 전체를 `/content/Aimers_9th`에 두거나 Google Drive에 마운트한 뒤, 아래 후보 경로에 실제 위치를 추가한다. `data/train.csv`, `src/features.py`, `experiments`가 함께 있어야 한다.

In [ ]:
from pathlib import Path
import json
import os
import subprocess
import shutil
import sys

try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
except ImportError:
    pass

candidates = [
    Path.cwd(),
    Path.cwd().parent,
    Path("/content/Aimers_9th"),
    Path("/content/drive/MyDrive/LG_aimers_9th"),
    Path("/content/drive/MyDrive/Aimers_9th"),
]
PROJECT_ROOT = next(
    (p.resolve() for p in candidates
     if (p / "data/train.csv").exists() and (p / "src/features.py").exists()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "프로젝트를 찾지 못했습니다. candidates에 Aimers_9th의 실제 경로를 추가하세요."
    )
print("PROJECT_ROOT =", PROJECT_ROOT)

### 1. 의존성 설치 및 런타임 확인

루트 `requirements.txt`는 로컬 재현 환경용이며 Drive 프로젝트 루트에도 보관한다. Colab은 NumPy·pandas·scikit-learn을 이미 로드할 수 있으므로 실행 중 강제 교체하지 않고, 현재 기준선에 추가로 필요한 LightGBM만 설치한다. 제출 ZIP의 `baseline_submit/requirements.txt`와는 목적이 다르다.

In [ ]:
requirements_path = PROJECT_ROOT / "requirements.txt"
if not requirements_path.is_file():
    raise FileNotFoundError(
        f"{requirements_path}가 없습니다. Drive의 LG_aimers_9th 루트에 requirements.txt를 업로드하세요."
    )

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    install_cmd = [
        sys.executable, "-m", "pip", "install",
        "--disable-pip-version-check", "lightgbm==4.6.0",
    ]
else:
    install_cmd = [
        sys.executable, "-m", "pip", "install",
        "--disable-pip-version-check", "-r", str(requirements_path),
    ]

install_result = subprocess.run(install_cmd, capture_output=True, text=True)
if install_result.returncode != 0:
    print(install_result.stdout)
    print(install_result.stderr)
    raise RuntimeError("패키지 설치에 실패했습니다. 위 pip 오류를 확인하세요.")
print(install_result.stdout.strip() or "Package requirements already satisfied.")

import lightgbm
import numpy
import pandas
import sklearn
print(
    "Versions:",
    f"lightgbm={lightgbm.__version__}",
    f"numpy={numpy.__version__}",
    f"pandas={pandas.__version__}",
    f"sklearn={sklearn.__version__}",
)
gpu_binary = shutil.which("nvidia-smi")
gpu = subprocess.run([gpu_binary], capture_output=True, text=True) if gpu_binary else None
print("GPU detected:", gpu is not None and gpu.returncode == 0)
if gpu is not None and gpu.returncode == 0:
    print(gpu.stdout.splitlines()[0])
print("Python:", sys.version.split()[0], "| CPU threads:", os.cpu_count())

### 2. 시간 홀드아웃 실행

기준선은 2019~2021→2022, 2019~2022→2023, 2019~2023→2024의 세 forward 폴드로 고정한다. 모든 피처는 현재 행과 학습 폴드에서 정해진 상수만 사용하며 다른 validation/test 행의 통계를 사용하지 않는다. 저장 결과가 현재 코드의 재현성 검사를 포함하지 않으면 자동으로 다시 실행한다.

In [ ]:
VALID_SEASONS = [2022, 2023, 2024]
THREADS = max(1, min(8, os.cpu_count() or 2))
FORCE_RERUN = False

for valid_season in VALID_SEASONS:
    output_dir = PROJECT_ROOT / "outputs" / f"lgbm_holdout_{valid_season}_train_R_no_ids"
    metrics_path = output_dir / "metrics.json"
    can_reuse = False
    if metrics_path.exists() and not FORCE_RERUN:
        with metrics_path.open(encoding="utf-8") as f:
            saved = json.load(f)
        can_reuse = (
            saved.get("reload_feature_frame_equal") is True
            and saved.get("reload_max_abs_diff") == 0.0
            and saved.get("train_game_type") == "R"
            and saved.get("include_player_ids") is False
            and saved.get("include_season") is True
            and saved.get("half_life") == 2.5
            and saved.get("parameters", {}).get("n_estimators") == 500
        )
    if can_reuse:
        print(f"[{valid_season}] 검증된 기존 결과 사용: {metrics_path}")
        continue
    if metrics_path.exists() and not FORCE_RERUN:
        print(f"[{valid_season}] 이전 형식/설정 결과를 새 형식으로 교체합니다.")

    cmd = [
        sys.executable, str(PROJECT_ROOT / "experiments/02_lgbm_holdout.py"),
        "--valid-season", str(valid_season),
        "--train-game-type", "R",
        "--exclude-player-ids",
        "--threads", str(THREADS),
        "--output-dir", str(output_dir),
    ]
    print("실행:", " ".join(cmd))
    subprocess.run(cmd, cwd=PROJECT_ROOT, check=True)

### 3. 폴드 결과 비교

공식 점수는 0 아래를 잘라내지만 실험에서는 음수도 원인을 보기 위해 `bss_raw`를 함께 기록한다. 모델 선택은 최신 한 폴드만 보지 말고 2022~2024의 Brier/BSS 방향과 평균 오차를 함께 본다.

In [ ]:
import pandas as pd

rows = []
for valid_season in VALID_SEASONS:
    path = PROJECT_ROOT / "outputs" / f"lgbm_holdout_{valid_season}_train_R_no_ids" / "metrics.json"
    with path.open(encoding="utf-8") as f:
        m = json.load(f)
    rows.append({
        "valid_season": valid_season,
        "train_rows": m["train_rows"],
        "brier": m["brier"],
        "bss_raw": m["bss_raw"],
        "bss_official": m["bss_official"],
        "roc_auc": m["roc_auc"],
        "target_rate": m["target_rate"],
        "prediction_mean": m["prediction_mean"],
        "prediction_std": m["prediction_std"],
        "mean_gap_pct_point": 100 * m["mean_gap"],
        "reload_max_abs_diff": m["reload_max_abs_diff"],
        "configuration_id": m["configuration_id"],
        "train_seconds": m["train_seconds"],
    })
summary = pd.DataFrame(rows).sort_values("valid_season")
summary

#### 3.1 통합 OOF 진단

세 검증 시즌의 OOF 예측을 행 단위로 합쳐 전체 Brier를 계산한다. 이 통합 BSS는 서로 다른 세 시즌을 합친 진단값이므로 DACON의 단일 2025 평가 점수와 동일하지 않다. 튜닝에서는 통합 Brier를 주 비교 지표로 사용하고 시즌별 Brier/BSS와 최신 2024 성능을 안전장치로 함께 본다.

In [ ]:
src_path = str(PROJECT_ROOT / "src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)
from evaluation import probability_metrics

oof_parts = []
for valid_season in VALID_SEASONS:
    fold_dir = PROJECT_ROOT / "outputs" / f"lgbm_holdout_{valid_season}_train_R_no_ids"
    part = pd.read_csv(fold_dir / "predictions.csv.gz")
    part["valid_season"] = valid_season
    oof_parts.append(part)
oof = pd.concat(oof_parts, ignore_index=True)
pooled = probability_metrics(oof["control_success"], oof["prediction"])
pooled_diagnostic = pd.Series({
    "oof_rows": pooled["n"],
    "pooled_brier": pooled["brier"],
    "pooled_diagnostic_bss": pooled["bss_raw"],
    "pooled_auc": pooled["roc_auc"],
    "target_rate": pooled["target_rate"],
    "prediction_mean": pooled["prediction_mean"],
    "mean_gap_pct_point": 100 * pooled["mean_gap"],
})
pooled_diagnostic

### 4. 취약 구간 확인

전체 점수만 좋아도 `game_type=F`, 다음 시즌 신규 선수, 누적 투구 수가 작은 구간에서 무너질 수 있다. 2022~2024의 모든 `group_metrics.csv`를 함께 확인한다. 하위 집단 BSS는 각 집단의 성공률을 분모로 다시 계산하므로 서로 단순 평균하거나 공식 전체 점수로 해석하지 않는다. 표본 1,000행 미만 집단은 경고만 확인하고 최적화 대상으로 삼지 않는다.

In [ ]:
group_parts = []
for valid_season in VALID_SEASONS:
    fold_dir = PROJECT_ROOT / "outputs" / f"lgbm_holdout_{valid_season}_train_R_no_ids"
    part = pd.read_csv(fold_dir / "group_metrics.csv")
    part["valid_season"] = valid_season
    part["mean_gap_pct_point"] = 100 * part["mean_gap"]
    group_parts.append(part)
group_metrics_all = pd.concat(group_parts, ignore_index=True)

core_groups = ["game_type", "pitcher_history", "batter_history", "cold_start"]
core_guardrails = group_metrics_all[group_metrics_all["group"].isin(core_groups)][[
    "valid_season", "group", "value", "n", "target_rate",
    "prediction_mean", "mean_gap_pct_point", "brier", "bss_raw", "roc_auc"
]].sort_values(["group", "value", "valid_season"])
core_guardrails

In [ ]:
segment_risk = (group_metrics_all[
    (~group_metrics_all["group"].eq("overall"))
    & (group_metrics_all["n"] >= 1_000)
][[
    "valid_season", "group", "value", "n",
    "mean_gap_pct_point", "brier", "bss_raw", "roc_auc"
]].sort_values(["bss_raw", "n"], ascending=[True, False]))
segment_risk.head(30)

#### 4.1 시즌별 캘리브레이션

각 폴드의 20분위 calibration table을 비교한다. `weighted_bin_abs_gap`은 진단용이며 공식 Brier를 대체하지 않는다. 현재 검증 시즌의 정답으로 예측을 보정하지 않고, 수준·퍼짐 오차가 어느 시즌에 반복되는지만 확인한다.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

calibration_parts = []
calibration_rows = []
for valid_season in VALID_SEASONS:
    fold_dir = PROJECT_ROOT / "outputs" / f"lgbm_holdout_{valid_season}_train_R_no_ids"
    part = pd.read_csv(fold_dir / "calibration.csv")
    part["valid_season"] = valid_season
    part["gap"] = part["prediction_mean"] - part["target_rate"]
    calibration_parts.append(part)
    calibration_rows.append({
        "valid_season": valid_season,
        "weighted_bin_abs_gap": np.average(part["gap"].abs(), weights=part["n"]),
        "max_bin_abs_gap": part["gap"].abs().max(),
    })
calibration_all = pd.concat(calibration_parts, ignore_index=True)
calibration_summary = pd.DataFrame(calibration_rows)
display(calibration_summary)

fig, ax = plt.subplots(figsize=(6.4, 5.2))
for valid_season, part in calibration_all.groupby("valid_season"):
    ax.plot(part["prediction_mean"], part["target_rate"], marker="o", label=str(valid_season))
lo = min(calibration_all["prediction_mean"].min(), calibration_all["target_rate"].min())
hi = max(calibration_all["prediction_mean"].max(), calibration_all["target_rate"].max())
ax.plot([lo, hi], [lo, hi], linestyle="--", color="gray", linewidth=1)
ax.set(xlabel="Prediction mean", ylabel="Observed rate", title="Out-of-time calibration by season")
ax.legend(title="Valid season")
ax.grid(alpha=0.25)
plt.show()

#### 4.2 피처 중요도의 시즌 안정성

폴드마다 gain 총량이 다르므로 폴드 내 비율로 정규화한 뒤 평균과 순위를 비교한다. 중요도가 높아도 시간 이동을 외운 피처일 수 있으므로, 이 표만으로 피처를 선택하지 않고 include/exclude 절제 실험으로 확인한다.

In [ ]:
importance_parts = []
for valid_season in VALID_SEASONS:
    fold_dir = PROJECT_ROOT / "outputs" / f"lgbm_holdout_{valid_season}_train_R_no_ids"
    part = pd.read_csv(fold_dir / "feature_importance.csv")
    part["valid_season"] = valid_season
    part["gain_share"] = part["importance_gain"] / part["importance_gain"].sum()
    part["rank"] = part["importance_gain"].rank(method="min", ascending=False)
    importance_parts.append(part)
importance_all = pd.concat(importance_parts, ignore_index=True)
importance_stability = (importance_all.groupby("feature")
    .agg(mean_gain_share=("gain_share", "mean"),
         mean_rank=("rank", "mean"),
         worst_rank=("rank", "max"),
         fold_count=("valid_season", "nunique"))
    .sort_values(["mean_gain_share", "mean_rank"], ascending=[False, True])
    .reset_index())
importance_stability.head(30)

#### 4.3 저장 모델 재현성 검사

각 실험 러너가 학습 후 `model.joblib`을 다시 로드하고, 저장된 범주형 맵과 피처 설정만으로 validation 입력을 재구성한다. 아래 `reload_max_abs_diff=0`은 저장 전후 예측이 완전히 같다는 뜻이다.

In [ ]:
import joblib

bundle_rows = []
required_bundle_keys = {
    "model", "feature_columns", "categorical_columns", "category_maps",
    "feature_config", "target_prior", "configuration_id",
}
for valid_season in VALID_SEASONS:
    fold_dir = PROJECT_ROOT / "outputs" / f"lgbm_holdout_{valid_season}_train_R_no_ids"
    with (fold_dir / "metrics.json").open(encoding="utf-8") as f:
        m = json.load(f)
    bundle = joblib.load(fold_dir / "model.joblib")
    bundle_rows.append({
        "valid_season": valid_season,
        "bundle_keys_ok": required_bundle_keys <= set(bundle),
        "reload_feature_frame_equal": m["reload_feature_frame_equal"],
        "reload_max_abs_diff": m["reload_max_abs_diff"],
        "model_size_mb": m["model_size_bytes"] / 1024**2,
        "python_version": m["python_version"],
        "lightgbm_version": m["package_versions"]["lightgbm"],
        "configuration_id": m["configuration_id"],
    })
bundle_checks = pd.DataFrame(bundle_rows)
assert bundle_checks["bundle_keys_ok"].all()
assert bundle_checks["reload_feature_frame_equal"].all()
assert bundle_checks["reload_max_abs_diff"].eq(0.0).all()
bundle_checks

### 5. 이 결과 다음에 할 일

1. 이 노트북의 세 폴드·통합 OOF·취약 구간·재현성 결과를 고정 기준선으로 사용한다.
2. LightGBM의 학습 기간, 최근 시즌 가중치, `game_type` 처리, `season` 및 ID 포함 여부를 먼저 비교한다.
3. 상위 데이터 구성에서만 트리 복잡도와 정규화 파라미터를 최적화한다.
4. LightGBM 설정을 고정한 뒤 Colab GPU를 사용하는 CatBoost/XGBoost OOF 예측을 만든다.
5. 모델별 시간 OOF 예측만으로 calibration과 constrained blend 가중치를 학습한다.
6. 최종 선택 후 2019~2024 전체로 재학습하고 DACON 코드 제출 ZIP을 만든다.

현재 validation 정답 평균으로 예측을 shift하거나, test 전체 `pred.mean()`을 목표값에 맞추는 보정은 하지 않는다.